## youtube video review

In [ ]:
# Chap 16. 개인프로젝트 유튜브 일반 영상 리뷰 다중 수집기

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time
import math
import pandas as pd
import random
import os
import re

print("=" *80)
print(" 개인프로젝트 유튜브 리뷰 (일반 동영상 다중 영상 수집)")
print("=" *80)
print("\n")

query_txt = input('1.크롤링할 유튜브의 키워드는 무엇입니까?: ')
query_txt = query_txt.replace('"','')

# 🌟 수정됨: 영상 1개 당 수집할 건수로 기준 변경
cnt = int(input('2.일반 "영상 1개 당" 수집할 댓글은 몇 건입니까?(예: 10): '))
page_cnt = math.ceil(cnt)

target_video_cnt = int(input('3.탐색할 일반 영상의 개수는 총 몇 개입니까?(예: 3): '))

f_dir = input("4.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\youtube 크롤링(개인 프로젝트)\\'

print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")

# 폴더 및 파일 생성
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+query_txt, exist_ok=True)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

# 크롬 드라이버 실행
s_time = time.time( )
# s = Service("c:/py_temp/chromedriver.exe") # 지우기
driver = webdriver.Chrome() # 서비스 경로 지정 없이 실행

driver.get("https://www.youtube.com/")
driver.maximize_window()
time.sleep(4)

# 유튜브 검색
keyword = driver.find_element(By.XPATH,"//*[@id='center']/yt-searchbox/div[1]/div/form/input")
for a in query_txt :
    keyword.send_keys(a)
    time.sleep(0.3)
time.sleep(1) 
driver.find_element(By.XPATH,'//*[@id="center"]/yt-searchbox/div[1]/button/span/span/div').click()
time.sleep(random.randrange(3,5))


# 누적 리스트 및 전체 카운트
y_URL2=[] 
reviewer2=[] 
review_d2=[] 
review2=[] 
like2=[] 
total_count = 0  # 🌟 새로 추가됨: 모든 영상 통합 누적 수집 건수

for i in range(target_video_cnt):
    print(f"\n{'=' * 80}")
    print(f"[{i+1}번째 유튜브 영상으로 진입합니다]")
    
    # [수정포인트 2] 찾을 기준을 전체 썸네일에서 -> 일반 동영상 영역 내 썸네일로 구체화 완료
    videos = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer a#thumbnail")
    valid_videos = [v for v in videos if v.get_attribute("href")]
    
    if i >= len(valid_videos):
        print("더 이상 클릭할 영상이 부족합니다.")
        break
        
    v = valid_videos[i]  
    href = v.get_attribute("href")
    print("클릭할 URL:", href)
    
    # 일반 click() 시 화면에 안 보이면 에러날 수 있어 자바스크립트 우회 클릭 사용
    driver.execute_script("arguments[0].click();", v)
    
    time.sleep(5)  

    # [수정포인트 3] 댓글창 여는 버튼 클릭 대신, 화면 스크롤 다운 기능으로 대체 완료
    driver.execute_script("window.scrollTo(0, 800);")
    time.sleep(4) # 댓글 로딩 대기

    # [수정포인트 4] 전체 정보 추출(BeautifulSoup) 및 헤더 태그에서 전체 댓글 수 조회하도록 변경 완료
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    count_tag = soup.select_one('h2#count yt-formatted-string.count-text')
    if count_tag:
        review_count_str = count_tag.get_text(strip=True).replace(",", "")
        result4 = re.search(r"\d+", review_count_str)
        search_cnt = int(result4.group()) if result4 else 0
    else:
        search_cnt = 0
        
    print(f"👉 이 영상의 전체 댓글 건수 : {search_cnt} 건")
    
    # 🌟 수정됨: 현재 영상에서 수집한 개수를 세는 변수 따로 생성
    video_comment_count = 0  
    prev_count = -1
    
    for a in range(1, page_cnt+1):
        print('리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')
        f = open(ff_name, 'a', encoding='UTF-8')

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        y_URL = driver.current_url

        reple_result = soup.find_all('ytd-comment-thread-renderer')

        for li in reple_result:
            main_comment = li.find('ytd-comment-view-model')
            if not main_comment:
                continue

            video_comment_count += 1
            total_count += 1
            print("\n")
            print(f"🚀 [영상 {i+1}] 목표 총 {cnt}건 중 {video_comment_count}번째 댓글 수집 중 =========")

            f.write("\n")
            f.write(f"[{i+1}번째 영상] 총 {cnt} 건 중 {video_comment_count} 번째 리뷰====\n")
            f.write("1.동영상 URL: " + y_URL + "\n")
            y_URL2.append(y_URL)
            time.sleep(0.5)

            try:
                reviewer = li.find('div', id='header-author').find('a', 'yt-simple-endpoint style-scope ytd-comment-view-model').get_text()
            except:
                reviewer = "알 수 없음"
            f.write("2.댓글작성자명: " + reviewer.replace("\n", "").strip() + "\n")
            reviewer2.append(reviewer)
            
            try:
                review_d = li.find('div', id='header-author').find('span', id='published-time-text').get_text()
            except:
                review_d = "알 수 없음"
            f.write("3.댓글작성일자: " + review_d.replace("\n", "").strip() + "\n")
            review_d2.append(review_d)

            try:
                review = li.find('div', id='content').find('span', 'yt-core-attributed-string yt-core-attributed-string--white-space-pre-wrap').get_text()
            except:
                review = ""
            f.write("4.리뷰내용: " + review + "\n")
            review2.append(review)

            try:
                like_node = li.select_one('like-button-view-model span[role="text"]')
                if like_node:
                    like = like_node.get_text(strip=True)
                else:
                    backup_node = li.select_one('#vote-count-middle')
                    like = backup_node.get_text(strip=True) if backup_node else '0'
                if not like:
                    like = '0'
            except:
                like = '0'
            f.write("5.좋아요횟수:" + like + "\n")
            like2.append(like)

            # 🛑 1개 영상에 대한 목표 건수 달성 시 내부 수집 중단
            if video_comment_count >= cnt:
                print(f"\n✅ 이번 영상에서의 수집 목표량({cnt}건) 달성 완료!")
                break

        time.sleep(0.2)
        f.close()

        # 더 이상 수집되는게(스크롤 효과) 없거나 목표치 넘기면 탈출
        if video_comment_count == prev_count or video_comment_count >= cnt:
            break
        prev_count = video_comment_count

        # [보너스 팁 추가] 반복 수집을 위해 맨 아래로 스크롤을 내려 숨겨진 댓글 로딩 (추가 완료)
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(2)

    # 4. 수집 완료 후 뒤로가기로 검색결과 화면 복귀
    print(f"\n[{i+1}번째 영상 댓글 수집 완료. 다음 영상을 찾기 위해 뒤로가기(driver.back) 합니다.]")
    driver.back()
    time.sleep(5)  # 검색결과 창 로딩을 넉넉히 대기


import pymysql

# 1. DB 연결
conn = pymysql.connect(
    host='localhost',         
    user='root',              
    password='Jx03151616~~',  # 본인 비밀번호
    db='youtube_db',  
    charset='utf8mb4',        
    cursorclass=pymysql.cursors.DictCursor
)

try:
    with conn.cursor() as cursor:
        
        # 2. 크롤링 데이터를 담을 '테이블' 만들기 (최초 1번만 만들어지고, 이미 있으면 에러 없이 넘어갑니다)
        create_table_sql = """
        CREATE TABLE IF NOT EXISTS youtube_video_reviews (
            id INT AUTO_INCREMENT PRIMARY KEY, -- 데이터 고유 번호 (1부터 자동 증가)
            video_url VARCHAR(255),            -- 동영상 주소
            reviewer VARCHAR(100),             -- 작성자 이름
            review_date VARCHAR(50),           -- 작성 일자
            review_text TEXT,                  -- 리뷰 내용 (길 수 있으므로 TEXT 설정)
            likes VARCHAR(50)                  -- 좋아요 숫자
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        cursor.execute(create_table_sql)
        
        # 3. 크롤링한 파이썬 리스트 데이터들을 DB에 한 줄씩 밀어넣기 (INSERT)
        insert_sql = """
        INSERT INTO youtube_reviews (video_url, reviewer, review_date, review_text, likes) 
        VALUES (%s, %s, %s, %s, %s)
        """
        
        # 리스트에 들어있는 개수만큼 반복하며 하나씩 저장합니다!
        for i in range(len(y_URL2)):
            # MySQL에 전송!
            cursor.execute(insert_sql, (
                y_URL2[i], 
                reviewer2[i], 
                review_d2[i], 
                review2[i], 
                like2[i]
            ))
            
        # 4. 작업을 마쳤다면 [최종 승인/저장] 명세 도장 쾅! (이걸 안 쓰면 DB에 반영되지 않습니다)
        conn.commit()
        
        print(f"🎉 짝짝짝! 총 {len(y_URL2)}개의 유튜브 리뷰 데이터가 MySQL에 완벽하게 저장되었습니다!")

except Exception as e:
    print("DB 연동/저장 중 오류 발생:", e)
    
finally:
    # 5. DB 연결 종료
    conn.close()


#Step 7. xls 형태와 csv 형태로 저장하기
news_reple = pd.DataFrame()
news_reple['동영상 URL']=pd.Series(y_URL2)
news_reple['댓글작성자명']=pd.Series(reviewer2)
news_reple['댓글 작성일자']=pd.Series(review_d2)
news_reple['리뷰내용']=pd.Series(review2)
news_reple['좋아요횟수']=pd.Series(like2)

news_reple.to_csv(fc_name,encoding="utf-8-sig",index=True)
news_reple.to_excel(fx_name ,index=True , engine='openpyxl')


# Step 8. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print(f"1.모든 작업 종료. 수집된 전체(누적) 리뷰수는 {total_count} 건 입니다.")
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)

driver.close()


 개인프로젝트 유튜브 리뷰 (일반 동영상 다중 영상 수집)


요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~

[1번째 유튜브 영상으로 진입합니다]
클릭할 URL: https://www.youtube.com/watch?v=EZaUIDa2g7c&pp=ygUM7Jis66as67iM7JiB
👉 이 영상의 전체 댓글 건수 : 511 건
리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~


🚀 [영상 1] 목표 총 30건 중 1번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 2번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 3번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 4번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 5번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 6번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 7번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 8번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 9번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 10번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 11번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 12번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 13번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 14번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 15번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 16번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 17번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30

🎉 짝짝짝! 총 40개의 유튜브 리뷰 데이터가 MySQL에 완벽하게 저장되었습니다!
